# 04 · Simulate a batch effect and correct it with scVI

In this tutorial, you will create two versions of DLPFC slice **151675**: a simulated baseline and a version with an added batch effect.
Both contain the same spots at the same locations. You will then use scVI to learn a shared representation of the two batches.

You only need `dlpfc/151675.h5ad`, the same input as notebooks 00 and 02.
Follow [setup](README.md#setup) for `scikit-misc` and the separate scvi-tools environment with CUDA.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import FEAST

TUTORIAL = Path.cwd() if Path.cwd().name == "tutorial" else Path.cwd() / "tutorial"
sys.path.insert(0, str(TUTORIAL))
from _utils import data_root, load_counts, gene_summary, gene_values, spatial_panel
DATA = data_root()

OUT = TUTORIAL / "outputs" / "04"
OUT.mkdir(parents=True, exist_ok=True)
print("FEAST", FEAST.__version__)

In [ ]:
reference = load_counts(DATA / "dlpfc/151675.h5ad", "ground_truth")

### 1. Generate two batches from the same slice

A batch effect changes expression measurements without changing which tissue spot they came from.
Here, we create a controlled example by adjusting gene mean, dispersion and the frequency of zero counts.

The values of `D` and `b` below define the adjustment. We reuse the fixed values from Study 04 so you do not need a second input file.
They were originally estimated from slices 151673 and 151508 with `FEAST.characterize_batch`.
Applying them to 151675 creates a specified perturbation, rather than estimating a natural batch difference for this slice.

Generate both versions from 151675 with seed **42**:

- `alpha=0.0` generates the baseline without the added effect. Its counts are still newly simulated.
- `alpha=1.0` applies the full adjustment.

Using the same seed helps limit unrelated differences between the two simulations.

<details>
<summary>Where these coefficients come from</summary>

The values are copied from `deformation_diagnostics.diagonal_affine` in
`FEAST_reproduce/04_batch_effect_removal/outputs/final_rerun_20260718_v2/simulations/provenance.json`.
They are embedded in this notebook, so neither that repository nor the original two slices is needed to run it.

</details>

In [ ]:
%%capture --no-stderr
deformation = FEAST.BatchDeformation(
    D=[0.9506859713168332, 0.5067732090337828, 0.9215737834857423],
    b=[-0.9666024309148558, 0.011108308402325857, 1.0822184425747938],
    name="study04_fixed_deformation_on_151675",
)
base = FEAST.simulate_batch_effect(reference, D=deformation.D, b=deformation.b, alpha=0.0, random_seed=42)
query = FEAST.simulate_batch_effect(reference, D=deformation.D, b=deformation.b, alpha=1.0, random_seed=42)

In [ ]:
assert base.obs_names.equals(query.obs_names)
assert np.array_equal(base.obsm["spatial"], query.obsm["spatial"])
pd.DataFrame({"D": deformation.D, "b": deformation.b},
             index=["log mean", "log dispersion", "logit zero proportion"])

### 2. Choose the same genes for both batches

Select 2,000 variable genes from the original 151675 slice and use them in both simulated batches.
scVI models raw counts, so keep the `counts` layer unnormalized.

When joining the batches, give each copy of a spot a unique name.
The `source_spot_id` column records which original spot it came from, allowing us to identify the matching pair later.

In [ ]:
import scanpy as sc
panel_source = reference.copy()
sc.pp.highly_variable_genes(panel_source, n_top_genes=2000, flavor="seurat_v3")
genes = panel_source.var_names[panel_source.var["highly_variable"]]
pair = ad.concat({"ref": base[:, genes], "query": query[:, genes]}, label="batch", index_unique="__")
pair.obs["source_spot_id"] = list(base.obs_names) + list(query.obs_names)
pair.layers["counts"] = pair.X.copy()
pair.write_h5ad(OUT / "batch_input.h5ad")

### 3. Run scVI

scVI uses the count matrix and batch labels to learn a shared representation of the spots.
The [scvi_step.py](scvi_step.py) script runs this step in its own environment and saves the result.
Tissue-layer labels and known spot pairs are reserved for evaluation.

We keep the Study 04 model settings: 10 latent dimensions, one hidden layer, a zero-inflated negative binomial (ZINB) count model,
and up to 400 training epochs with early stopping.
The output includes the learned representation and normalized expression adjusted to the reference batch.

In [ ]:
import os
import subprocess
with (OUT / "scvi.log").open("w") as log:
    subprocess.run([
        os.environ["SCVI_PYTHON"], str(TUTORIAL / "scvi_step.py"),
        str(OUT / "batch_input.h5ad"), str(OUT / "scvi_result.h5ad"),
    ], check=True, stdout=log, stderr=subprocess.STDOUT)
corrected = ad.read_h5ad(OUT / "scvi_result.h5ad")

### 4. Compare batch mixing and tissue layers

Plot the spots before and after correction, coloring them first by batch and then by tissue layer.
The batch-colored plots show whether the two simulated batches overlap.
The layer-colored plots show whether the representation still separates the tissue layers.

For display, we use two principal components of normalized expression before correction and two principal components of the scVI representation afterward.
These are different coordinate systems, so compare their patterns rather than their axis values.

In [ ]:
from sklearn.decomposition import PCA
before = corrected.copy()
before.X = before.layers["counts"].copy()
sc.pp.normalize_total(before, target_sum=1e4)
sc.pp.log1p(before)
sc.pp.pca(before, n_comps=10, random_state=42)
projections = {"Before (count PCA)": before.obsm["X_pca"][:, :2],
               "After (scVI projection)": PCA(n_components=2, random_state=42).fit_transform(corrected.obsm["X_scVI"])}
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
for col, (title, xy) in enumerate(projections.items()):
    for row, key in enumerate(["batch", "ground_truth"]):
        spatial_panel(axes[row, col], xy, corrected.obs[key], f"{title} · {key}", True)
plt.show()

**How to read the plots:** look for batch mixing together with preserved tissue-layer structure.
Mixing alone is not enough if distinct layers have also merged.
These plots provide a visual check, rather than a numerical score of correction quality.

Because both batches come from one slice, their spot matches are known.
Performance on naturally different sections is a separate question, explored in Study 04's real-section experiment.

### Run status

The 151675 example has not yet been run end to end. Run the cells above to generate the paired batches, scVI result and plots.